# SOB4ES — 03. Unión y Estandarización Final
### CRISP-ML(Q) · Fase 2: Ingeniería de Datos

Este notebook combina las salidas de los dos notebooks anteriores en un único dataset listo para modelado.

| Paso | Descripción |
|------|-------------|
| **1** | Configuración y rutas |
| **2** | Cargar salidas de notebooks 01 y 02 |
| **3** | Corregir nombres de columna rotos (snake_case sobre acrónimos) |
| **4** | Unir datos locales + online |
| **5** | Imputar NaN de las variables online |
| **6** | Escalar variables online con el mismo scaler del notebook 01 |
| **7** | Exportar dataset final combinado |

**Nota: **La x en vx se refiere a la versión/iteración del archivo, esto se aplica a todos los outputs empleados.

**Entradas**: `sob4es_clean_vx.csv`, `sob4es_model_ready_vx.csv`, `online_features.csv`, `scaler.pkl`, `label_encoders.pkl`  
**Salidas**: `sob4es_final_clean.csv`, `sob4es_final_model_ready.csv`

## 0.- Configuración

In [16]:
import os, sys, warnings, re
import numpy  as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

OUT_DIR = 'output/'

os.makedirs(OUT_DIR, exist_ok=True)

# Rutas de entrada
CLEAN_PATH   = OUT_DIR + 'sob4es_clean_v11.csv'
MODEL_PATH   = OUT_DIR + 'sob4es_model_ready_v11.csv'
ONLINE_PATH  = OUT_DIR + 'online_features_v2.csv'
SCALER_PATH  = OUT_DIR + 'scaler.pkl'
ENCODER_PATH = OUT_DIR + 'label_encoders.pkl'

for nombre, ruta in [
    ('CLEAN_PATH',   CLEAN_PATH),
    ('MODEL_PATH',   MODEL_PATH),
    ('ONLINE_PATH',  ONLINE_PATH),
    ('SCALER_PATH',  SCALER_PATH),
    ('ENCODER_PATH', ENCODER_PATH),
]:
    existe = os.path.exists(ruta)
    print(f'  {nombre:15s}: {ruta}  {"Encontrado..." if existe else "[ERROR] Archivo no encontrado, por favor revisa las rutas..."}')

  CLEAN_PATH     : output/sob4es_clean_v11.csv  Encontrado...
  MODEL_PATH     : output/sob4es_model_ready_v11.csv  Encontrado...
  ONLINE_PATH    : output/online_features_v2.csv  Encontrado...
  SCALER_PATH    : output/scaler.pkl  Encontrado...
  ENCODER_PATH   : output/label_encoders.pkl  Encontrado...


## 1.- Cargar Salidas de los Notebooks Anteriores

In [17]:
df_clean  = pd.read_csv(CLEAN_PATH)
df_model  = pd.read_csv(MODEL_PATH)
df_online = pd.read_csv(ONLINE_PATH)

print(f'clean      : {df_clean.shape}')
print(f'model_ready: {df_model.shape}')
print(f'online     : {df_online.shape}')
print()
print('Online columns:', list(df_online.columns))

clean      : (428, 108)
model_ready: (428, 95)
online     : (428, 7)

Online columns: ['SITE_ID', 'gee_temp_media_C', 'gee_humedad_rel_pct', 'gee_ndvi_verano', 'dem_elevacion_m', 'dem_pendiente_deg', 'dem_orientacion_deg']


## 2.- Corregir Nombres de Columna Rotos

La función `estandarizar_nombre()` del notebook 01 aplicó snake_case sobre columnas
que ya eran acrónimos en mayúsculas (ej. `SITE_ID`, `BACTERIA_SHANNON`, `CN`).
El resultado: cada letra separada por guiones bajos (`s_i_t_e_i_d`, `b_a_c_t_e_r_i_a...`).

Se corrigen aquí con un mapeo explícito antes de hacer cualquier unión.

In [18]:
assert 'SITE_ID' in df_clean.columns, 'SITE_ID no encontrado en clean'
assert 'SITE_ID' in df_model.columns, 'SITE_ID no encontrado en model_ready'
assert 'SITE_ID' in df_online.columns, 'SITE_ID no encontrado en online'

# Verificar que no quedan nombres rotos del patrón letra_letra_letra
broken = [c for c in df_clean.columns
          if re.search(r'(?<![a-z])([a-z])_([a-z])_([a-z])', c)]
if broken:
    print(f'[WARN] Columnas con patrón roto detectadas: {broken}')
    print('  → Añadir entradas al RENAME_MAP si las versiones no coinciden.')
else:
    print(f'Nombres de columna verificados ({df_clean.shape[1]} cols en clean)  ✓')
print(f'SITE_ID: {df_clean["SITE_ID"].nunique()} sitios únicos en clean')

Nombres de columna verificados (108 cols en clean)  ✓
SITE_ID: 428 sitios únicos en clean


## 3.- Unir Datos Locales + Online

In [19]:
# Unir clean + online por SITE_ID
df_full = df_clean.merge(df_online, on='SITE_ID', how='left', suffixes=('', '_online'))

assert len(df_full) == len(df_clean), f'Pérdida de filas: {len(df_clean)} → {len(df_full)}'
print(f'Clean  : {df_clean.shape}')
print(f'Online : {df_online.shape}')
print(f'Unido  : {df_full.shape}')
print()

# Cobertura de variables online
online_feat_cols = [c for c in df_online.columns if c != 'SITE_ID']
cobertura_online = (
    df_full[online_feat_cols].notna().sum() / len(df_full) * 100
).round(1)
print('Cobertura variables online:')
for col, pct in cobertura_online.items():
    print(f'  {col:30s}  {pct:.1f}%')

Clean  : (428, 108)
Online : (428, 7)
Unido  : (428, 114)

Cobertura variables online:
  gee_temp_media_C                99.8%
  gee_humedad_rel_pct             99.8%
  gee_ndvi_verano                 97.9%
  dem_elevacion_m                 99.1%
  dem_pendiente_deg               99.1%
  dem_orientacion_deg             99.1%


In [20]:
assert 'SITE_ID' in df_model.columns, \
    'SITE_ID no encontrado en model_ready — verificar que data-prep.ipynb es v10+'

print(f'SITE_ID en model_ready: {df_model["SITE_ID"].nunique()} sitios únicos  ✓')
print(f'Shape model_ready: {df_model.shape}')

SITE_ID en model_ready: 428 sitios únicos  ✓
Shape model_ready: (428, 95)


## 4.- Imputar NaN en Variables Online

Misma estrategia que en el notebook 01:
- Variables continuas < 5% NaN: mediana
- Variables continuas 5-50% NaN: columna `_was_missing` + mediana
- \> 50% NaN → eliminar

In [21]:
# Solo aplicar a las columnas online (prefijos gee_, cds_, dem_)
online_num_cols = [c for c in online_feat_cols
                   if df_full[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

miss_pct = df_full[online_num_cols].isnull().mean() * 100

# Eliminar columnas > 50% NaN
drop_online = miss_pct[miss_pct > 50].index.tolist()
if drop_online:
    print(f'Eliminando {len(drop_online)} columnas online con >50% NaN: {drop_online}')
    df_full = df_full.drop(columns=drop_online)
    online_num_cols = [c for c in online_num_cols if c not in drop_online]
    miss_pct = miss_pct.drop(index=drop_online)

# Indicadores _was_missing para columnas 5-50% NaN
cols_indicator = miss_pct[(miss_pct >= 5) & (miss_pct <= 50)].index.tolist()
for col in cols_indicator:
    df_full[col + '_was_missing'] = df_full[col].isna().astype(int)
print(f'Indicadores _was_missing añadidos para variables online: {len(cols_indicator)}')

# Imputar con mediana
medians_online = df_full[online_num_cols].median()
df_full[online_num_cols] = df_full[online_num_cols].fillna(medians_online)

remaining = df_full[online_num_cols].isnull().sum().sum()
print(f'NaN restantes en variables online: {remaining}')
print(f'Shape tras imputación: {df_full.shape}')

Indicadores _was_missing añadidos para variables online: 0
NaN restantes en variables online: 0
Shape tras imputación: (428, 114)


## 5.- Escalar Variables Online

Las variables online (`gee_*`, `cds_*`, `dem_*`) no estaban en el dataset cuando se ajustó
el `scaler.pkl` del notebook 01. 

Se ajusta un scaler **nuevo** solo para estas columnas y se guarda por separado para poder aplicarlo en inferencia.

In [22]:
# Columnas online a escalar (excluir _was_missing — son binarias)
online_scale_cols = [
    c for c in online_num_cols
    if not c.endswith('_was_missing')
]

scaler_online = StandardScaler()
scaled_online = scaler_online.fit_transform(df_full[online_scale_cols])
df_scaled_online = pd.DataFrame(
    scaled_online,
    columns=[c + '_z' for c in online_scale_cols],
    index=df_full.index
)

joblib.dump(scaler_online, OUT_DIR + 'scaler_online.pkl')
print(f'Escaladas {len(online_scale_cols)} variables online')
print(f'Scaler guardado en {OUT_DIR}scaler_online.pkl')
print(f'Columnas escaladas: {list(df_scaled_online.columns)}')

Escaladas 6 variables online
Scaler guardado en output/scaler_online.pkl
Columnas escaladas: ['gee_temp_media_C_z', 'gee_humedad_rel_pct_z', 'gee_ndvi_verano_z', 'dem_elevacion_m_z', 'dem_pendiente_deg_z', 'dem_orientacion_deg_z']


## 6.- Ensamblar y Exportar Dataset Final

In [23]:
# 1. Dataset final limpio (legible, escala original)
df_final_clean = df_full.copy()

# Reorganizamos los datos
bloque_geo   = ['SITE_ID', 'Country', 'Pedoclimatic_region', 'Site_locality', 'latitude', 'longitude', 'Sampling_date']
bloque_desc  = ['soil_type', 'Land_use_type', 'Land_use_intensity', 'Dominant_vegetation']

cols_totales = list(df_final_clean.columns)

# Abióticos y fisicoquímica del suelo (Ground Truth de campo)
cols_abiotic = [c for c in cols_totales if c in [
    'Total_Plant_cover', 'Bulk density', 'Soil moisture', 'aggregate_stability', 'clay_content', 'silt_content', 'sand_content',
    'soil_pH', 'plot_Total_C', 'plot_Total_organic_C', 'plot_Total_N', 'P', 'K', 'As', 'Cu', 'Mo', 'Ni', 'Pb', 'Zn'
]]

# Índices globales de diversidad alfa
cols_shannon = [c for c in cols_totales if 'shannon' in c.lower()]

# Comunidades de fauna (Macro, Meso y Micro) ordenadas taxonómicamente
cols_fauna   = [c for c in cols_totales if c.startswith(('macro_', 'Earthworm_', 'orib_', 'meso_', 'coll_')) and c not in cols_shannon]

# Microbioma y secuenciación molecular (ASVs y lecturas)
cols_micro   = [c for c in cols_totales if c.startswith(('bac_', 'fun_', 'euk_', 'oomy_', 'cerc_')) and c not in cols_shannon]

# Teledetección y Modelos Digitales de Terreno (GEE / CDS / DEM)
cols_gee_dem = [c for c in cols_totales if c.startswith(('gee_', 'cds_', 'dem_'))]

# Proxies espaciales de la Unión Europea (Capas Raster EU)
cols_eu      = [c for c in cols_totales if c.startswith('eu_') and not c.endswith('_enc')]

# Codificaciones categóricas presentes en clean
cols_enc     = [c for c in cols_totales if c.endswith('_enc')]

# Meta-indicadores
cols_meta    = ['outlier_flag'] if 'outlier_flag' in cols_totales else []

# Ensamblar el nuevo ordenamiento lógico
nuevo_orden_clean = (
    bloque_geo + bloque_desc + cols_abiotic + cols_shannon + 
    cols_fauna + cols_micro + cols_gee_dem + cols_eu + cols_meta + cols_enc
)

# Conservamos solo columnas existentes evitando duplicados
nuevo_orden_clean = [c for c in nuevo_orden_clean if c in df_final_clean.columns]

# Si quedó alguna columna fuera, se anexa al final
columnas_huerfanas_clean = [c for c in df_final_clean.columns if c not in nuevo_orden_clean]
if columnas_huerfanas_clean:
    nuevo_orden_clean += columnas_huerfanas_clean

# Reindexamos el DataFrame
df_final_clean = df_final_clean[nuevo_orden_clean]

# Exportación del archivo limpio
final_clean_path = OUT_DIR + 'sob4es_final_clean.csv'
df_final_clean.to_csv(final_clean_path, index=False)
print(f'{final_clean_path} done...')
print(f'Tamaño: {df_final_clean.shape[0]} sitios × {df_final_clean.shape[1]} columnas')

output/sob4es_final_clean.csv done...
Tamaño: 428 sitios × 114 columnas


In [24]:
# 2. Dataset final listo para modelo.

df_model_final = (
    df_model
    .merge(
        pd.concat([df_full[['SITE_ID']], df_scaled_online], axis=1),
        on='SITE_ID', how='left'
    )
)

# Nos aseguramos que no se creen ni eliminen filas
assert len(df_model_final) == len(df_model), \
    f'Explosión de filas: {len(df_model)} → {len(df_model_final)} — comprobar SITE_IDs duplicados'


# Reorganizamos las columnas

id_cols = ['SITE_ID', 'latitude', 'longitude']
nuevo_orden_model = id_cols.copy()

# Generamos la estructura basada exactamente en las posiciones relativas de 'nuevo_orden_clean'
for c in nuevo_orden_clean:
    if c in id_cols:
        continue
    # Incorporamos la versión estandarizada si existe en este dataset
    if f"{c}_z" in df_model_final.columns:
        nuevo_orden_model.append(f"{c}_z")
    # Incorporamos la versión codificada si existe en este dataset
    if f"{c}_enc" in df_model_final.columns:
        nuevo_orden_model.append(f"{c}_enc")
    # Incorporamos la variable tal cual si ya residía ahí (ej. outlier_flag)
    if c in df_model_final.columns and c not in nuevo_orden_model:
        nuevo_orden_model.append(c)

# Purgamos cualquier _was_missing residual que pudiera venir arrastrado desde df_model original
nuevo_orden_model = [c for c in nuevo_orden_model if not str(c).endswith('_was_missing')]

# Cualquier otra columna técnica remanente se mantiene al final
columnas_huerfanas_model = [c for c in df_model_final.columns if c not in nuevo_orden_model and not str(c).endswith('_was_missing')]
if columnas_huerfanas_model:
    nuevo_orden_model += columnas_huerfanas_model

# Reindexamos el DataFrame del Modelo
df_model_final = df_model_final[nuevo_orden_model]

# Exportación del archivo para modelado
final_model_path = OUT_DIR + 'sob4es_final_model_ready.csv'
df_model_final.to_csv(final_model_path, index=False)
print(f'{final_model_path} done...')
print(f'Tamaño: {df_model_final.shape[0]} sitios × {df_model_final.shape[1]} columnas')

# Desglose estadístico final del archivo del modelo
z_cols    = [c for c in df_model_final.columns if c.endswith('_z')]
enc_cols  = [c for c in df_model_final.columns if c.endswith('_enc')]
wm_cols   = [c for c in df_model_final.columns if c.endswith('_was_missing')]
print(f'Desglose: {len(z_cols)} _z  |  {len(enc_cols)} _enc  |  {len(wm_cols)} _was_missing')

output/sob4es_final_model_ready.csv done...
Tamaño: 428 sitios × 101 columnas
Desglose: 87 _z  |  10 _enc  |  0 _was_missing


## 7.- Resumen Final

In [ ]:
def inferir_fuente(col):
    if col.startswith('gee_'):               return 'GEE (online)'
    if col.startswith('cds_'):               return 'CDS (online)'
    if col.startswith('dem_'):               return 'DEM (online)'
    if col.startswith('eu_'):                return 'Raster EU'
    if col.startswith('macro_'):             return 'Macrofauna'
    if col.startswith('orib_'):              return 'Oribátida'
    if col.startswith('meso_'):              return 'Mesostigmata'
    if col.startswith('coll_'):              return 'Colémbolos'
    if col.startswith('bac_'):               return 'Bacterias (16S)'
    if col.startswith('fun_'):               return 'Hongos (ITS)'
    if col.startswith('euk_'):               return 'Eucariotas (18S)'
    if col.startswith('oomy_'):              return 'Oomycetes'
    if col.startswith('cerc_'):              return 'Cercozoa'
    if col.startswith('plot_'):              return 'Abiótico (parcela)'
    if col in {'clay_content','silt_content','sand_content',
               'aggregate_stability','Bulk density','Soil moisture'}:
        return 'Abiótico (físico)'
    if col in {'As','Cu','K','Mo','Ni','P','Pb','Zn','soil_pH'}:
        return 'Abiótico (químico)'
    if 'Shannon' in col or 'SHANNON' in col or 'Richness' in col \
            or 'Abundance' in col or 'asv_' in col or 'reads' in col:
        return 'Diversidad alfa'
    if col in {'SITE_ID','latitude','longitude','Country',
               'Pedoclimatic_region','Sampling_date','Site_locality',
               'soil_type','Land_use_type','Land_use_intensity',
               'Dominant_vegetation','Total_Plant_cover'}: return 'Metadatos sitio'
    if col.endswith('_z'):             return 'Escalado (z)'
    if col.endswith('_enc'):           return 'Codificado (enc)'
    if col.endswith('_was_missing'):  return 'Indicador NaN'
    return 'Otro'

catalogo = pd.DataFrame({
    'dtype'    : df_final_clean.dtypes,
    'n_nulos'  : df_final_clean.isnull().sum(),
    'n_unicos' : df_final_clean.nunique(),
    'fuente'   : [inferir_fuente(c) for c in df_final_clean.columns],
})

print('    Columnas por fuente    ')
print(catalogo.groupby('fuente').size().sort_values(ascending=False)
      .rename('n_columnas').to_string())
print()
print('    Resumen final    ')
print(f'Sitios                : {df_final_clean.shape[0]}')
print(f'Columnas (clean)      : {df_final_clean.shape[1]}')
print(f'Columnas (model_ready): {df_model_final.shape[1]}')
print(f'NaN totales (clean)   : {df_final_clean.isnull().sum().sum()}')
print(f'NaN totales (model)   : {df_model_final.isnull().sum().sum()}')

--- Columnas por fuente ---
fuente
Macrofauna            22
Raster EU             19
Diversidad alfa       12
Metadatos sitio       12
Abiótico (químico)     9
Codificado (enc)       6
Abiótico (físico)      6
Colémbolos             4
Abiótico (parcela)     3
GEE (online)           3
DEM (online)           3
Eucariotas (18S)       2
Bacterias (16S)        2
Cercozoa               2
Hongos (ITS)           2
Mesostigmata           2
Oomycetes              2
Oribátida              2
Otro                   1

--- RESUMEN FINAL ---
Sitios                : 428
Columnas (clean)      : 114
Columnas (model_ready): 101
NaN totales (clean)   : 0
NaN totales (model)   : 0
